In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
select * from yipidata.bronze.company_metadata

company,employee_count,founded_year,headquarters,industry,is_public,stock_ticker
Airbnb,19967,1999,"London, UK",Data Analytics,false,null
Amazon Web Services,27826,2015,"Berlin, Germany",SaaS,false,null
Anthropic,43747,2006,"San Francisco, CA",FinTech,false,null
Cloudflare,4422,1998,"Austin, TX",Cybersecurity,false,CLOU
Confluent,3884,1995,"London, UK",Cloud Computing,false,null
DataRobot,23471,2012,"New York, NY",Cloud Computing,true,null
Databricks,17962,2003,"Berlin, Germany",Data Analytics,true,DATA
Elastic,24508,2016,"San Francisco, CA",Cybersecurity,true,ELAS
Google DeepMind,24274,2017,"Austin, TX",Data Analytics,false,null
Meta AI,20240,2001,"Seattle, WA",Data Analytics,true,null


In [0]:
df = spark.read.table('yipidata.bronze.company_metadata')

In [0]:
df = df.withColumn('company_size', when(col('employee_count') < 10000, 'Small')\
              .when((col('employee_count') >= 10000) & (col('employee_count') <= 30000), 'Medium')\
              .when(col('employee_count') > 30000, 'Large')\
              .otherwise('')
)

In [0]:
df.display()

company,employee_count,founded_year,headquarters,industry,is_public,stock_ticker,company_size
Airbnb,19967,1999,"London, UK",Data Analytics,false,null,Medium
Amazon Web Services,27826,2015,"Berlin, Germany",SaaS,false,null,Medium
Anthropic,43747,2006,"San Francisco, CA",FinTech,false,null,Large
Cloudflare,4422,1998,"Austin, TX",Cybersecurity,false,CLOU,Small
Confluent,3884,1995,"London, UK",Cloud Computing,false,null,Small
DataRobot,23471,2012,"New York, NY",Cloud Computing,true,null,Medium
Databricks,17962,2003,"Berlin, Germany",Data Analytics,true,DATA,Medium
Elastic,24508,2016,"San Francisco, CA",Cybersecurity,true,ELAS,Medium
Google DeepMind,24274,2017,"Austin, TX",Data Analytics,false,null,Medium
Meta AI,20240,2001,"Seattle, WA",Data Analytics,true,null,Medium


In [0]:
#use a merge here
#merge on company

In [0]:
df.printSchema()

root
 |-- company: string (nullable = true)
 |-- employee_count: long (nullable = true)
 |-- founded_year: long (nullable = true)
 |-- headquarters: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- is_public: boolean (nullable = true)
 |-- stock_ticker: string (nullable = true)
 |-- company_size: string (nullable = false)



In [0]:
%sql
CREATE TABLE IF NOT EXISTS yipidata.silver.company_metadata (
    company STRING,
    employee_count BIGINT,
    founded_year BIGINT,
    headquarters STRING,
    industry STRING,
    is_public BOOLEAN,
    stock_ticker STRING,
    company_size STRING 
)
USING DELTA


In [0]:
df.createOrReplaceTempView('source_view')

In [0]:
df.display()

company,employee_count,founded_year,headquarters,industry,is_public,stock_ticker,company_size
Airbnb,19967,1999,"London, UK",Data Analytics,false,null,Medium
Amazon Web Services,27826,2015,"Berlin, Germany",SaaS,false,null,Medium
Anthropic,43747,2006,"San Francisco, CA",FinTech,false,null,Large
Cloudflare,4422,1998,"Austin, TX",Cybersecurity,false,CLOU,Small
Confluent,3884,1995,"London, UK",Cloud Computing,false,null,Small
DataRobot,23471,2012,"New York, NY",Cloud Computing,true,null,Medium
Databricks,17962,2003,"Berlin, Germany",Data Analytics,true,DATA,Medium
Elastic,24508,2016,"San Francisco, CA",Cybersecurity,true,ELAS,Medium
Google DeepMind,24274,2017,"Austin, TX",Data Analytics,false,null,Medium
Meta AI,20240,2001,"Seattle, WA",Data Analytics,true,null,Medium


In [0]:
%sql
MERGE INTO yipidata.silver.company_metadata AS trg
USING source_view AS src 
ON trg.company = src.company
WHEN MATCHED THEN 
UPDATE SET *
WHEN NOT MATCHED 
THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
21,0,0,21


In [0]:
%sql
SELECT * FROM  yipidata.silver.company_metadata

company,employee_count,founded_year,headquarters,industry,is_public,stock_ticker,company_size
Airbnb,19967,1999,"London, UK",Data Analytics,false,null,Medium
Amazon Web Services,27826,2015,"Berlin, Germany",SaaS,false,null,Medium
Anthropic,43747,2006,"San Francisco, CA",FinTech,false,null,Large
Cloudflare,4422,1998,"Austin, TX",Cybersecurity,false,CLOU,Small
Confluent,3884,1995,"London, UK",Cloud Computing,false,null,Small
DataRobot,23471,2012,"New York, NY",Cloud Computing,true,null,Medium
Databricks,17962,2003,"Berlin, Germany",Data Analytics,true,DATA,Medium
Elastic,24508,2016,"San Francisco, CA",Cybersecurity,true,ELAS,Medium
Google DeepMind,24274,2017,"Austin, TX",Data Analytics,false,null,Medium
Meta AI,20240,2001,"Seattle, WA",Data Analytics,true,null,Medium
